# Assignment 4 — End-to-end: raw file to model-ready data

**Dataset:** UFO sightings (≈80k reports of the National UFO Reporting Center, via TidyTuesday 2019-06-25).
**Source link:** https://github.com/rfordatascience/tidytuesday/tree/master/data/2019/2019-06-25
(raw CSV: https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-06-25/ufo_sightings.csv)

**Prediction task this data is prepared for:** classify the reported `ufo_shape` from when, where and how the sighting was reported. **No model is trained here**: the notebook stops at a clean matrix (`X_train, X_test, y_train, y_test`).

Why this dataset fits the brief: 11 columns, 80,332 rows, mixed types (datetime, categories, free text, numeric), genuine missing values, and real mess: HTML-escaped text, impossible timestamps, durations recorded in two conflicting ways, and country codes that are missing even when the state is known.

**Notebook layout.** The four required functions each sit in their own cell (marked **[Q4]**). Q1–Q3 are analysis that uses them; the final cell (Q5) runs the full path.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import re, html, time
import numpy as np, pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)

### [Q4] Function 1 of 4 — `load_data()`

In [2]:
URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-06-25/ufo_sightings.csv"

def load_data():
    "Read the raw file exactly as published (no cleaning)."
    return pd.read_csv(URL)

raw = load_data()
print("raw shape:", raw.shape)
raw.head(3)

raw shape: (80332, 11)


,date_time,city_area,state,country,ufo_shape,encounter_length,described_encounter_length,description,date_documented,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 1949-50. It occurred after a Boy ...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing across the sky &amp; making 90 degr...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 England,1/21/2008,53.200000,-2.916667


## Q1. Data dictionary
| column | meaning | type | unit | expected range |
|---|---|---|---|---|
| `date_time` | local date and time of the sighting (text `m/d/Y H:M` in the file) | datetime | local time | 1900–2014; hour 0–23 |
| `city_area` | city or area of the sighting, sometimes with a region/country in brackets | text | – | free text |
| `state` | US state or Canadian province code | category | 2-letter code | ~50 US + ~13 Canadian codes; blank outside US/Canada |
| `country` | country code | category | lower-case 2-letter | `us`, `ca`, `gb`, `au`, `de` |
| `ufo_shape` | shape chosen by the witness (**target**) | category | – | ~20 labels |
| `encounter_length` | how long the sighting lasted | float | seconds | > 0 and ≤ 86,400 (24 h) |
| `described_encounter_length` | the same duration in the witness's own words | text | mixed (sec/min/hours) | free text |
| `description` | summary of the report, cut off by the source at ~250 characters | text | characters | 1–250 |
| `date_documented` | date the report was filed | date | – | ≥ 1998-03 and ≥ `date_time`, ≤ 2014 |
| `latitude` | latitude of the sighting | float | degrees | −90 to 90, and inside the stated country |
| `longitude` | longitude of the sighting | float | degrees | −180 to 180, and inside the stated country |

### Helpers shared by the quality report and by `clean()` (constants and small pure functions)

In [3]:
US_STATES = set("al ak az ar ca co ct de dc fl ga hi id il in ia ks ky la me md ma mi mn ms mo mt ne nv nh nj nm ny nc nd oh ok or pa ri sc sd tn tx ut vt va wa wv wi wy pr".split())
CA_PROVINCES = set("ab bc mb nb nf nl ns nt nu on pe pq qc sk yk yt".split())
# rough bounding boxes (lat_min, lat_max, lon_min, lon_max) of the countries present in the data
BBOX = {"us": (17, 72, -180, -64), "ca": (41, 84, -142, -52), "gb": (49, 61, -9, 2.5),
        "au": (-45, -9, 112, 155), "de": (47, 55.5, 5.5, 15.5)}
SHAPE_MAP = {"round": "circle", "changed": "changing", "delta": "triangle",
             "pyramid": "triangle", "flare": "other", "hexagon": "other", "dome": "other", "crescent": "other"}
MAX_SECONDS = 24 * 3600
DEDUP_KEYS = ["date_time", "city_area", "description"]
TEXT_COLS = ["city_area", "state", "country", "ufo_shape"]
_NUM = r"(\d+\s+\d+/\d+|\d+/\d+|\d*\.\d+|\d+)"
_DUR = re.compile(rf"(?:about|approx\.?|approximately|around|roughly|~)?\s*{_NUM}\s*(seconds?|secs?|s|minutes?|mins?|hours?|hrs?|h)\.?")
_FACTOR = {"s": 1, "m": 60, "h": 3600}

def _to_number(tok):
    tok = tok.strip()
    if " " in tok:                                   # mixed fraction "1 1/2"
        whole, frac = tok.split(); a, b = frac.split("/"); return int(whole) + int(a) / int(b)
    if "/" in tok:
        a, b = tok.split("/"); return int(a) / int(b)
    return float(tok)

def parse_duration_text(s):
    "Seconds from simple texts such as '2 min', '1/2 hour', '1 1/2 hrs'. NaN for anything ambiguous (ranges, '?', words)."
    def one(t):
        if not isinstance(t, str): return np.nan
        m = _DUR.fullmatch(t.strip().lower())
        return _to_number(m.group(1)) * _FACTOR[m.group(2)[0]] if m else np.nan
    return s.map(one)

def parse_sighting_datetime(s):
    "Parse m/d/Y H:M. The source writes midnight as 24:00, which is the *next* day at 00:00."
    s = s.str.strip()
    is24 = s.str.endswith(" 24:00")
    dt = pd.to_datetime(s.str.replace(" 24:00", " 00:00", regex=False), format="%m/%d/%Y %H:%M", errors="coerce")
    return dt.where(~is24, dt + pd.Timedelta(days=1))

def fix_html(s):
    "Undo HTML escaping: '&#44' -> ',', '&#39s' -> \"'s\", '&amp;' -> '&'."
    s = s.str.replace(r"&#(\d\d)", lambda m: chr(int(m.group(1))), regex=True)
    return s.map(lambda v: html.unescape(v) if isinstance(v, str) else v)

HTML_PAT = r"&#\d\d|&(?:amp|quot|lt|gt);"

def fill_country(df):
    "Country from state, using hard-coded domain knowledge (US states, Canadian provinces)."
    df = df.copy()
    df.loc[df["country"].isna() & df["state"].isin(US_STATES), "country"] = "us"
    df.loc[df["country"].isna() & df["state"].isin(CA_PROVINCES), "country"] = "ca"
    return df

def coord_mismatch(df):
    "True where a coordinate lies outside the bounding box of the row's own country."
    bad_all = pd.Series(False, index=df.index)
    for c, (la0, la1, lo0, lo1) in BBOX.items():
        outside = (df.latitude < la0) | (df.latitude > la1) | (df.longitude < lo0) | (df.longitude > lo1)
        bad_all |= (df.country == c) & outside
    return bad_all

def quality_report(raw):
    "One row per problem found in the raw file, with a count."
    n, rows = len(raw), []
    def add(kind, problem, count): rows.append({"type": kind, "problem": problem, "rows affected": int(count), "% of rows": round(100 * count / n, 2)})
    for c in raw.columns[raw.isna().any()]:
        add("missing values", f"{c} is missing", raw[c].isna().sum())
    add("duplicates", "exact duplicate rows", raw.duplicated().sum())
    add("duplicates", "same sighting twice (date_time + city_area + description)", raw.duplicated(DEDUP_KEYS).sum())
    strict = pd.to_datetime(raw.date_time, format="%m/%d/%Y %H:%M", errors="coerce")
    fixed = parse_sighting_datetime(raw.date_time)
    add("impossible values", "date_time is not a valid timestamp (hour 24:00)", strict.isna().sum())
    add("impossible values", "  ...of which still unparseable after the 24:00 fix", fixed.isna().sum())
    doc = pd.to_datetime(raw.date_documented, format="%m/%d/%Y", errors="coerce")
    add("impossible values", "report filed before the sighting happened", (doc < fixed.dt.normalize()).sum())
    add("impossible values", "encounter_length <= 0", (raw.encounter_length <= 0).sum())
    add("impossible values", "encounter_length > 24 hours", (raw.encounter_length > MAX_SECONDS).sum())
    add("impossible values", "latitude outside [-90, 90] or longitude outside [-180, 180]", ((raw.latitude.abs() > 90) | (raw.longitude.abs() > 180)).sum())
    tmp = fill_country(raw.assign(state=raw.state.str.strip().str.lower(), country=raw.country.str.strip().str.lower()))
    add("impossible values", "coordinates outside the country (country filled from state first)", coord_mismatch(tmp).sum())
    parsed = parse_duration_text(raw.described_encounter_length)
    ratio = raw.encounter_length / parsed
    add("inconsistent units", "numeric seconds disagree (>1.5x) with the witness's own simple text, e.g. '1/2 hour' stored as 20 s", ((ratio > 1.5) | (ratio < 1 / 1.5)).sum())
    st = raw.state.isin(US_STATES | CA_PROVINCES)
    add("inconsistent labels", "country missing although state is a US/Canadian code", (raw.country.isna() & st).sum())
    add("inconsistent labels", "ufo_shape variants/rare synonyms (round, changed, delta, pyramid, flare, hexagon, dome, crescent: 16 rows in total, each too rare to stratify on)", raw.ufo_shape.isin(SHAPE_MAP).sum())
    add("inconsistent labels", "ufo_shape is 'unknown' or 'other' (ambiguous, kept as labels)", raw.ufo_shape.isin(["unknown", "other"]).sum())
    add("inconsistent labels", "HTML-escaped characters in description (&#44, &amp; ...)", raw.description.str.contains(HTML_PAT, regex=True, na=False).sum())
    add("inconsistent labels", "HTML-escaped characters in city_area", raw.city_area.str.contains(HTML_PAT, regex=True, na=False).sum())
    ws = pd.Series(False, index=raw.index)
    for c in TEXT_COLS: ws |= raw[c].notna() & (raw[c] != raw[c].str.strip().str.lower())
    add("inconsistent labels", "stray whitespace or upper-case in text columns", ws.sum())
    return pd.DataFrame(rows)

## Q2. Raw quality report (every problem, with a count)

In [4]:
report = quality_report(raw)
print("Total missing cells in the raw file:", int(raw.isna().sum().sum()))
report

Total missing cells in the raw file: 17418


,type,problem,rows affected,% of rows
0,missing values,state is missing,5797,7.22
1,missing values,country is missing,9670,12.04
2,missing values,ufo_shape is missing,1932,2.41
3,missing values,encounter_length is missing,3,0.00
4,missing values,description is missing,15,0.02
5,missing values,latitude is missing,1,0.00
6,duplicates,exact duplicate rows,0,0.00
7,duplicates,same sighting twice (date_time + city_area + description),9,0.01
8,impossible values,date_time is not a valid timestamp (hour 24:00),694,0.86
9,impossible values,...of which still unparseable after the 24:00 fix,0,0.00


## Q3. New features and why

| feature | built from | one-line reason |
|---|---|---|
| `hour_sin`, `hour_cos` | hour of `date_time` | The hour is circular (23:00 and 00:00 are neighbours); sin/cos keep that, a raw 0–23 number would not. |
| `is_night` | hour of `date_time` | Lights and fireballs are only visible in the dark, so time of day should separate shapes. |
| `year`, `month` | `date_time` | Reporting habits and popular "shapes" changed over the decades, and summer evenings produce more sightings. |
| `log_duration` | `encounter_length` | Raw seconds are extremely right-skewed (skew > 100); `log1p` makes a 10 s flash and a 10 min hover comparable. |
| `log_report_lag` | `date_documented − date_time` | Reports filed years later rely on fading memory, so the label is less reliable. |
| `desc_words` | `description` | A longer, more detailed report is a proxy for how carefully the witness observed the object. |

### [Q4] Function 2 of 4 — `clean(df)` (deterministic row-level rules only, nothing is learned from the data)

In [5]:
def clean(df):
    "Fix what can be fixed without learning anything from the data; learned steps (imputing, scaling) live in the pipeline."
    df = df.copy()
    # 1. text hygiene: whitespace, case, HTML escapes
    for c in TEXT_COLS: df[c] = df[c].str.strip().str.lower()
    df["city_area"] = fix_html(df["city_area"])
    df["description"] = fix_html(df["description"]).str.strip()
    # 2. dates: 24:00 means next day 00:00; filing before the sighting is impossible -> missing
    df["date_time"] = parse_sighting_datetime(df["date_time"])
    df["date_documented"] = pd.to_datetime(df["date_documented"], format="%m/%d/%Y", errors="coerce")
    df = df[df["date_time"].notna()]
    df.loc[df["date_documented"] < df["date_time"].dt.normalize(), "date_documented"] = pd.NaT
    # 3. duration: where the witness's simple text disagrees with the stored seconds, trust the text; then cap impossible values
    parsed = parse_duration_text(df["described_encounter_length"])
    ratio = df["encounter_length"] / parsed
    disagree = (ratio > 1.5) | (ratio < 1 / 1.5)
    df.loc[disagree, "encounter_length"] = parsed[disagree]
    df.loc[(df["encounter_length"] <= 0) | (df["encounter_length"] > MAX_SECONDS), "encounter_length"] = np.nan
    # 4. labels: merge synonyms; the target must exist, so rows without a shape are dropped
    df["ufo_shape"] = df["ufo_shape"].replace(SHAPE_MAP)
    df = df[df["ufo_shape"].notna()]
    # 5. country from state (domain knowledge, hard-coded lists)
    df = fill_country(df)
    # 6. coordinates that contradict the country cannot be trusted -> drop the row
    df = df[~coord_mismatch(df)]
    # 7. duplicates: same sighting entered twice
    df = df.drop_duplicates(subset=DEDUP_KEYS, keep="first")
    # 8. columns that are fully captured elsewhere
    return df.drop(columns=["described_encounter_length"]).reset_index(drop=True)

### [Q4] Function 3 of 4 — `add_features(df)`

In [6]:
def add_features(df):
    out = df.copy()
    hour = out["date_time"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["is_night"] = ((hour >= 20) | (hour < 5)).astype(int)
    out["year"] = out["date_time"].dt.year
    out["month"] = out["date_time"].dt.month
    out["log_duration"] = np.log1p(out["encounter_length"])
    lag_days = (out["date_documented"] - out["date_time"].dt.normalize()).dt.days
    out["log_report_lag"] = np.log1p(lag_days)
    out["desc_words"] = out["description"].fillna("").str.split().str.len()
    return out

### [Q4] Function 4 of 4 — `build_pipeline()` (learned steps: imputing, encoding, scaling; fitted on train only)

In [7]:
NUMERIC = ["year", "month", "hour_sin", "hour_cos", "is_night", "log_duration",
           "log_report_lag", "desc_words", "latitude", "longitude"]
CATEGORICAL = ["country", "state"]

def build_pipeline():
    numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),   # + flag columns where train had gaps
                        ("scale", StandardScaler())])
    categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False))])
    return Pipeline([("prep", ColumnTransformer([("num", numeric, NUMERIC), ("cat", categorical, CATEGORICAL)], remainder="drop"))])

## Q5. Full path: raw file → `X_train, X_test, y_train, y_test`
The split happens **after** `clean()` and `add_features()` (both are row-wise and learn nothing) and **before** the pipeline is fitted, so every imputer, encoder and scaler sees training rows only.

In [8]:
raw = load_data()
cleaned = add_features(clean(raw))
X_df, y = cleaned.drop(columns=["ufo_shape"]), cleaned["ufo_shape"]
X_train_df, X_test_df, y_train, y_test = train_test_split(X_df, y, test_size=0.2, stratify=y, random_state=SEED)

pipeline = build_pipeline()
X_train = pipeline.fit_transform(X_train_df)      # fitted on train only
X_test = pipeline.transform(X_test_df)
feature_names = pipeline.get_feature_names_out()

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("\nTarget classes:", y.nunique(), "| most common share: %.1f%%" % (100 * y.value_counts(normalize=True).iloc[0]))

X_train: (62649, 77)
X_test:  (15663, 77)
y_train: (62649,)
y_test:  (15663,)

Target classes: 21 | most common share: 21.1%


## Q6. Proof the output is clean

In [9]:
assert not np.isnan(X_train).any() and not np.isnan(X_test).any(), "NaN remains"
assert X_train.shape[1] == X_test.shape[1], "train/test column counts differ"
assert (X_train.std(axis=0) > 0).all(), "a column is constant in train"
assert len(y_train) == len(X_train) and len(y_test) == len(X_test), "row counts differ"
assert y_train.notna().all() and y_test.notna().all(), "missing target"
print("All assertions passed: no NaN, %d columns in both train and test, no constant column." % X_train.shape[1])

All assertions passed: no NaN, 77 columns in both train and test, no constant column.


## Q7. Before and after

In [10]:
full_matrix = np.vstack([X_train, X_test])
before = {"rows": len(raw), "columns": raw.shape[1], "missing values": int(raw.isna().sum().sum()),
          "exact duplicate rows": int(raw.duplicated().sum()), "duplicate sightings (key columns)": int(raw.duplicated(DEDUP_KEYS).sum())}
mid = {"rows": len(cleaned), "columns": cleaned.shape[1], "missing values": int(cleaned.isna().sum().sum()),
       "exact duplicate rows": int(cleaned.duplicated().sum()), "duplicate sightings (key columns)": int(cleaned.duplicated(DEDUP_KEYS).sum())}
after = {"rows": len(full_matrix), "columns": full_matrix.shape[1], "missing values": int(np.isnan(full_matrix).sum()),
         "exact duplicate rows": int(pd.DataFrame(full_matrix).duplicated().sum()), "duplicate sightings (key columns)": "n/a (keys removed)"}
ba = pd.DataFrame({"Raw file": before, "Cleaned DataFrame": mid, "Model-ready matrix (train+test)": after})
ba

,Raw file,Cleaned DataFrame,Model-ready matrix (train+test)
rows,80332,78312,78312
columns,11,18,77
missing values,17418,9153,0
exact duplicate rows,0,0,54
duplicate sightings (key columns),9,0,n/a (keys removed)


*How to read the last column:* the matrix has no key columns left, so "duplicates" there means rows whose whole feature vector is identical. The missing values left in the cleaned DataFrame are deliberate: they are the ones the pipeline fills with statistics learned from training data only (`country`, `state`, `log_duration`, ...), plus the free-text fields it does not use.

## Q8. Save the cleaned data and the fitted pipeline

In [11]:
cleaned.to_csv("ufo_cleaned.csv", index=False)
joblib.dump(pipeline, "pipeline.joblib")

# reload and confirm the saved pipeline reproduces the in-memory result
reloaded = joblib.load("pipeline.joblib")
assert np.allclose(reloaded.transform(X_test_df.head(100)), X_test[:100])
import os
print("ufo_cleaned.csv: %.1f MB | pipeline.joblib: %.1f KB | reload check passed" % (os.path.getsize("ufo_cleaned.csv") / 1e6, os.path.getsize("pipeline.joblib") / 1e3))

ufo_cleaned.csv: 19.6 MB | pipeline.joblib: 6.0 KB | reload check passed


In [12]:
try:
    from google.colab import files
    files.download("ufo_cleaned.csv"); files.download("pipeline.joblib")
except ImportError:
    print("Not running in Colab: both files are in the working directory.")

Not running in Colab: both files are in the working directory.


## Q9. Every decision and the number that drove it

Row accounting: **80,332 raw rows → 78,312 kept (−2,020, 2.5%)**; 11 raw columns → 77 model columns.

- Dropped **1,932 rows (2.41%)** with no `ufo_shape`: the target cannot be missing, and imputing it would invent labels.
- Fixed **694 timestamps (0.86%)** written as `24:00` by rolling them to 00:00 of the next day; 0 remained unparseable, so no rows were lost.
- Merged **16 rows** with one-off shape labels (`round`→circle, `changed`→changing, `delta`/`pyramid`→triangle, `flare`/`hexagon`/`dome`/`crescent`→other): each label had 1–7 rows, and the stratified split crashed on a class with a single member.
- Kept `unknown` and `other` as their own classes (**11,233 rows, 14.0%**): the witness chose these labels, so they are answers, not missing data.
- Filled `country` from `state` for **6,413 rows (7.98%)** (5,813 US, 600 Canada) using fixed lists of US states and Canadian provinces; **3,182 rows (4.1%)** stay missing and get a `missing` category in the pipeline.
- Replaced **300 durations (0.37%)** where the stored seconds disagreed by more than 1.5× with the witness's own simple text (e.g. "1/2 hour" stored as 20 s) by the value parsed from the text.
- Set **161 durations > 24 h** to missing (179 in the raw file; 144 are left in the kept rows): a witness cannot watch a sighting for days. The pipeline median-imputes them and adds a missing-flag column.
- Dropped **79 rows (0.10%)** whose coordinates fall outside their own country's bounding box (e.g. "jersey, nj, us" at 49°N, 2°W).
- Dropped **9 duplicate sightings (0.01%)** (same `date_time` + `city_area` + `description`), before the split so a duplicate can never sit on both sides.
- Set **8 `date_documented` values** (filed before the sighting) to missing, so their `log_report_lag` is imputed by the median with a flag.
- Un-escaped HTML in **33,511 descriptions (41.7%)** and **1,221 city names**, so word counts are not inflated by `&#44`.
- Dropped `city_area` (**19,900 unique values** in 80k rows, almost an ID), `described_encounter_length` (redundant once reconciled), and the raw dates and text after turning them into features.
- Took `log1p` of duration because skew was **117.7** raw, **15.0** after the fixes above and **−0.12** after `log1p`.
- One-hot encoded `state` and `country` with `min_frequency=100`: **11 of 68 state** and **1 of 6 country** categories are pooled into an "infrequent" bucket, which is also where unseen categories land.
- Median imputation plus missing-flags for the **3 numeric columns** that had gaps in train (`log_duration`, `log_report_lag`, `latitude`); median because the raw numbers are heavy-tailed.
- Split **80/20, stratified on 21 classes, `random_state=42`** → 62,649 train / 15,663 test, and fitted the pipeline on train only.

## Q10. What is still wrong with this dataset

The biggest remaining problem is the target itself: `ufo_shape` is what a witness chose from a menu, with no ground truth, and 14% of rows say `unknown` or `other`, so there is a ceiling on how well any model can predict it. The data is also a self-selected sample, not a census: 88% of reports are from the US, 45% are from 2008 onward, and every report filed before 1998 (13% of sightings) was documented retrospectively, so `year` and `log_report_lag` are tangled together. Durations are witness estimates that cluster on round values (75% are whole minutes or 30/45 s), and I could only cross-check the 73% of duration texts simple enough to parse, so ranges like "4-8 minutes" could not be verified. Descriptions are cut off at about 246 characters, so I only use their length and not the text. Coordinates that are inside their country's bounding box can still be wrong (they look geocoded from the city name rather than measured), and I have no way to check that. Finally, 3,182 rows have no country and 5,658 have no state, and for non-US/Canadian rows the missing state is partly a signal rather than a gap, which the `missing` category encodes only crudely.